# Strands Agents with Bedrock AgentCore Browser — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Browser to give your AI agent the ability to navigate websites, extract data, and monitor regulatory updates.

## Overview

In this lab, you will:
- Connect to a remote browser session via AgentCore
- Navigate financial websites and extract data
- Monitor regulatory sites (APRA, ASX) for updates
- Compare bank rates programmatically

## Why Browser Automation for FSI?

- **Regulatory monitoring** — Check APRA, ASIC, ASX for policy changes
- **Market data extraction** — Scrape rates, prices from financial portals
- **Competitor analysis** — Compare product rates across banks
- **Compliance evidence** — Screenshot proof of checks performed

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore playwright

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## Part 1: Create a Custom Browser with Public Network

The default browser (`aws.browser.v1`) has restrictive settings. For accessing external sites like regulatory portals, we create a **custom browser** with:
- Public network access (can reach any website)
- Browser signing (helps with bot detection on some sites)

In [ ]:
from bedrock_agentcore._utils import endpoints
import boto3
from botocore.exceptions import ClientError

region = boto3.session.Session().region_name
cp_endpoint = endpoints.get_control_plane_endpoint(region)
dp_endpoint = endpoints.get_data_plane_endpoint(region)

cp_client = boto3.client('bedrock-agentcore-control', region_name=region, endpoint_url=cp_endpoint)
dp_client = boto3.client('bedrock-agentcore', region_name=region, endpoint_url=dp_endpoint)

# Create a custom browser with public network access
browser_name = 'fsi_regulatory_browser'
try:
    response = cp_client.create_browser(
        name=browser_name,
        description='Custom browser for FSI regulatory site monitoring',
        networkConfiguration={'networkMode': 'PUBLIC'},
        
    )
    browser_id = response['browserId']
    print(f'✅ Custom browser created: {browser_id}')
except ClientError as e:
    if 'already exists' in str(e).lower() or 'Conflict' in str(e):
        browsers = cp_client.list_browsers()['browserSummaries']
        browser_id = next(b['browserId'] for b in browsers if b.get('name') == browser_name)
        print(f'✅ Using existing browser: {browser_id}')
    else:
        raise e

print(f'   Network: PUBLIC')
print(f'   Signing: Enabled (helps with bot detection)')

### Test the Custom Browser with Playwright

Let's start a session and navigate to APRA (Australian Prudential Regulation Authority):

## Part 2: Strands Agent with Browser Automation

Now let's give a Strands Agent browser capabilities. The agent will navigate to Amazon, search for a product, and extract information — demonstrating autonomous web interaction.

**Real-world FSI applications:**
- Navigating internal portals and legacy web apps
- Filling compliance forms on regulatory sites
- Extracting data from web-based dashboards without APIs

In [ ]:
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright

# Use our custom browser (with public network)
with browser_session(region, identifier=browser_id) as client:
    print(f'🌐 Session: {client.session_id}')
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        print('✅ Browser connected')

        context = browser.contexts[0] if browser.contexts else await browser.new_context()
        page = context.pages[0] if context.pages else await context.new_page()

        # Navigate to APRA
        await page.goto('https://www.apra.gov.au/news-and-publications')
        await page.wait_for_load_state('domcontentloaded')

        title = await page.title()
        content = await page.text_content('body')
        print(f'Page: {title}')
        print(f'Content: {content[:300]}...')

        await browser.close()

## Part 3: APRA Information Papers

Let's check for recent APRA information papers — a common compliance monitoring task.

In [ ]:
# Check ASIC for BNPL regulatory updates
response = await browser_agent.invoke_async(
    'Go to https://www.apra.gov.au/information-papers and list any recent information papers.'
)
console.print(response.message['content'][0].get('text', ''))

## Part 4: RBA Cash Rate

Check the Reserve Bank of Australia for the current cash rate target.

In [ ]:
# Check RBA for cash rate
response = await browser_agent.invoke_async(
    'Go to https://www.rba.gov.au/statistics/cash-rate/ and tell me the current cash rate target.'
)
console.print(response.message['content'][0].get('text', ''))

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f"Number of Loops: {browser_agent.event_loop_metrics.cycle_count}")
console.print(f"Messages in conversation: {len(browser_agent.messages)}")

## Cleanup

In [ ]:
# Clean up all browser sessions
# client = boto3.client('bedrock-agentcore')
# args = {'browserIdentifier': 'aws.browser.v1', 'status': 'READY'}
# response = client.list_browser_sessions(**args)
# for session in response['items']:
#     client.stop_browser_session(browserIdentifier='aws.browser.v1', sessionId=session['sessionId'])
# print('✅ Browser sessions cleaned up')

## Common FSI Use Cases for Browser Automation

| Use Case | Example |
|----------|--------|
| Regulatory monitoring | Check APRA/ASIC/ASX for new publications |
| Rate comparison | Compare home loan rates across banks |
| Market data | Extract ASX indices, stock prices |
| KYC/AML checks | Verify entities against public registries |
| Compliance evidence | Screenshot proof of monitoring activities |
| Competitor analysis | Track competitor product changes |

## Summary

In this lab, you:

- ✅ Created a remote browser session via AgentCore
- ✅ Connected with Playwright for direct browser control
- ✅ Used a Strands Agent to navigate financial websites autonomously
- ✅ Monitored regulatory sites (APRA) for updates
- ✅ Compared bank rates programmatically

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Autonomous navigation | Agent checks regulatory sites without manual effort |
| Data extraction | Structured data from unstructured web pages |
| Screenshot capture | Compliance evidence of monitoring activities |
| Secure environment | Isolated browser — no risk to internal systems |

### Next: Lab 04 — AgentCore Runtime MCP
We'll deploy a transaction validation tool as a managed MCP server with authentication.